In [8]:
import pandas as pd

# Carga del dataset
df = pd.read_parquet('limpio_consumohdna_20251-20253.parquet')

In [ ]:
# Modelo estrella
# ---- Dim_Cliente ----
dim_cliente = (
    df[['DEPARTAMENTO', 'PROVINCIA', 'DISTRITO']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_cliente.insert(0, 'id_cliente', dim_cliente.index + 1)

# ---- Dim_Tarifa ----
dim_tarifa = (
    df[['TARIFA', 'CARTERA']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_tarifa.insert(0, 'id_tarifa', dim_tarifa.index + 1)

# ---- Dim_Sucursal ----
dim_sucursal = (
    df[['UNIDAD_NEGOCIO']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_sucursal.insert(0, 'id_sucursal', dim_sucursal.index + 1)

# ---- Dim_Fecha ----
dim_fecha = (
    df[['FECHA']]
    .drop_duplicates()
    .sort_values('FECHA')
    .reset_index(drop=True)
)
dim_fecha.insert(0, 'id_fecha', dim_fecha.index + 1)
dim_fecha['ANIO'] = dim_fecha['FECHA'].dt.year
dim_fecha['MES'] = dim_fecha['FECHA'].dt.month
dim_fecha['NOMBRE_MES'] = dim_fecha['FECHA'].dt.month_name()

# ---- Fact_Factura ----
fact_factura = (
    df.merge(dim_cliente, on=['DEPARTAMENTO', 'PROVINCIA', 'DISTRITO'], how='left')
      .merge(dim_tarifa, on=['TARIFA', 'CARTERA'], how='left')
      .merge(dim_sucursal, on=['UNIDAD_NEGOCIO'], how='left')
      .merge(dim_fecha[['id_fecha', 'FECHA']], on='FECHA', how='left')
)
fact_factura = fact_factura[['id_cliente', 'id_tarifa', 'id_sucursal', 'id_fecha', 'IMPORTE', 'CONSUMO']]

print('Dim_Cliente :', dim_cliente.shape)
print('Dim_Tarifa  :', dim_tarifa.shape)
print('Dim_Sucursal:', dim_sucursal.shape)
print('Dim_Fecha   :', dim_fecha.shape)
print('Fact_Factura:', fact_factura.shape)

Dim_Cliente : (334, 4)
Dim_Tarifa  : (12, 3)
Dim_Sucursal: (7, 2)
Dim_Fecha   : (3, 4)
Fact_Factura: (3100476, 6)


In [10]:
# Conexión a SQL Server
import sqlalchemy as sa
from sqlalchemy import text
import urllib

# ---- Parámetros de conexión ----
SERVER = 'localhost'
DATABASE = 'HidrandinaDW' # la base de datos debe existir previamente
DRIVER = 'ODBC Driver 18 for SQL Server'

# ---- Autenticación de Windows (sin usuario y contraseña) ----
params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;Encrypt=no;"
)

engine = sa.create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT @@VERSION")).scalar()
    print('Conexión exitosa a SQL Server:')
    print(version)

Conexión exitosa a SQL Server:
Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 19045: ) (Hypervisor)



In [ ]:
# Creación de tablas en SQL Server
ddl = """
IF OBJECT_ID('dbo.Fact_Factura', 'U') IS NOT NULL DROP TABLE dbo.Fact_Factura;
IF OBJECT_ID('dbo.Dim_Cliente', 'U') IS NOT NULL DROP TABLE dbo.Dim_Cliente;
IF OBJECT_ID('dbo.Dim_Tarifa', 'U') IS NOT NULL DROP TABLE dbo.Dim_Tarifa;
IF OBJECT_ID('dbo.Dim_Sucursal', 'U') IS NOT NULL DROP TABLE dbo.Dim_Sucursal;
IF OBJECT_ID('dbo.Dim_Fecha', 'U') IS NOT NULL DROP TABLE dbo.Dim_Fecha;

CREATE TABLE dbo.Dim_Cliente (
    id_cliente     INT PRIMARY KEY,
    DEPARTAMENTO   VARCHAR(100),
    PROVINCIA      VARCHAR(100),
    DISTRITO       VARCHAR(100)
);

CREATE TABLE dbo.Dim_Tarifa (
    id_tarifa      INT PRIMARY KEY,
    TARIFA         VARCHAR(50),
    CARTERA        VARCHAR(50)
);

CREATE TABLE dbo.Dim_Sucursal (
    id_sucursal    INT PRIMARY KEY,
    UNIDAD_NEGOCIO VARCHAR(100)
);

CREATE TABLE dbo.Dim_Fecha (
    id_fecha       INT PRIMARY KEY,
    FECHA          DATE,
    ANIO           INT,
    MES            INT,
    NOMBRE_MES     VARCHAR(20)
);

CREATE TABLE dbo.Fact_Factura (
    id_cliente     INT NOT NULL,
    id_tarifa      INT NOT NULL,
    id_sucursal    INT NOT NULL,
    id_fecha       INT NOT NULL,
    IMPORTE        DECIMAL(18,2),
    CONSUMO        DECIMAL(18,2),
    CONSTRAINT FK_Factura_Cliente  FOREIGN KEY (id_cliente)  REFERENCES dbo.Dim_Cliente(id_cliente),
    CONSTRAINT FK_Factura_Tarifa   FOREIGN KEY (id_tarifa)   REFERENCES dbo.Dim_Tarifa(id_tarifa),
    CONSTRAINT FK_Factura_Sucursal FOREIGN KEY (id_sucursal) REFERENCES dbo.Dim_Sucursal(id_sucursal),
    CONSTRAINT FK_Factura_Fecha    FOREIGN KEY (id_fecha)    REFERENCES dbo.Dim_Fecha(id_fecha)
);
"""

with engine.begin() as conn:
    for statement in ddl.split(';'):
        if statement.strip():
            conn.execute(text(statement))

print('Tablas creadas correctamente en SQL Server.')

Tablas creadas correctamente en SQL Server.


In [12]:
# Carga de datos
dim_cliente.to_sql('Dim_Cliente', engine, schema='dbo', if_exists='append', index=False, chunksize=1000)
dim_tarifa.to_sql('Dim_Tarifa', engine, schema='dbo', if_exists='append', index=False, chunksize=1000)
dim_sucursal.to_sql('Dim_Sucursal', engine, schema='dbo', if_exists='append', index=False, chunksize=1000)
dim_fecha.to_sql('Dim_Fecha', engine, schema='dbo', if_exists='append', index=False, chunksize=1000)
fact_factura.to_sql('Fact_Factura', engine, schema='dbo', if_exists='append', index=False, chunksize=1000)

print('Carga completada en SQL Server.')

Carga completada en SQL Server.


In [ ]:
# Verificación de la carga
with engine.connect() as conn:
    for tabla in ['Dim_Cliente', 'Dim_Tarifa', 'Dim_Sucursal', 'Dim_Fecha', 'Fact_Factura']:
        n = conn.execute(text(f"SELECT COUNT(*) FROM dbo.{tabla}")).scalar()
        print(f"{tabla}: {n} filas")

Dim_Cliente: 334 filas
Dim_Tarifa: 12 filas
Dim_Sucursal: 7 filas
Dim_Fecha: 3 filas
Fact_Factura: 3100476 filas
